# Annulus twists of $6_3$

## Libraries

In [1]:
import sys; sys.path.append("../modules")
from Ideal_elm_pair_class import *

import itertools
#from snappy import *
#S = twister.Surface('S_2_1')

import matplotlib.pyplot as plt
%display latex

# Basic data of $6_3$

## md_data class

In [2]:
class md_data(ideal):
    def __init__(self, md:str):
        super().__init__(md)
        self.ugens = self._del_dup_and_sort(self.UnitGroup.gens_values())
        self.Cgens = self._del_dup_and_sort([ug*(self.tau(ug)) for ug in self.ugens])

    def show(self):
        print("\n-- Symp. Matrix --")
        display(self.matrix)
        print("\n-- Number field --")
        display(self.field)
        print("\n-- basis of OK --")
        display(self.field.ring_of_integers().basis())
        print("\n-- class number of OK --")
        display(self.field.ring_of_integers().class_number())
        print("\n-- basis of U --")
        display(self.ugens)
        print("\n-- basis of C --")
        display(self.Cgens)

    def get_C(self, R:int=10):
        C = set()
        r = len(self.Cgens)
        signs = sorted(list(itertools.product(range(-1,2), repeat=r)), key=functools.cmp_to_key(self._compare_tuples))
        #---
        for ids in itertools.product(range(R+1), repeat=r):
            for s in signs:
                factors = [self.Cgens[i]**(s[i]*ids[i]) for i in range(r)]
                u = functools.reduce(lambda x,y: x*y, factors)
                if u == self.tau(u):
                    C.add(u)
        return C

    def _del_dup_and_sort(self, mlist:list):
        return sorted(list(set(mlist)), key=lambda x: x.polynomial().degree())

    def _compare_tuples(self, a,b):
        absed = [list(map(lambda x: abs(x), t)) for t in [a,b]]
        sums = [functools.reduce(lambda x,y: x^2+y^2, t) for t in absed]
        (A, B) = (a, b) if sums[0] == sums[1] else (sums[0], sums[1])
        if A > B: 
            return 1
        elif A < B: 
            return -1
        else: 
            return 0        

## Data

In [3]:
m63 = 'DbCa'
D63 = md_data(m63)
D63.show()


-- Symp. Matrix --


[ 0  1 -1 -1]
[-1  1  0  0]
[ 1 -1  1  0]
[ 0  0  1  1]


-- Number field --


Number Field in alpha with defining polynomial t^4 - 3*t^3 + 5*t^2 - 3*t + 1


-- basis of OK --


...

1


-- basis of U --


[alpha^3 - 3*alpha^2 + 4*alpha - 1, -alpha^3 + 2*alpha^2 - 3*alpha]


-- basis of C --


[-alpha^3 + 3*alpha^2 - 4*alpha + 2, alpha^3 - 3*alpha^2 + 4*alpha - 2]

In [4]:
print(f"-- elements of C --")
C63 = D63.get_C(R=50); [display(v) for v in C63];

-- elements of C --


1

alpha^3 - 3*alpha^2 + 4*alpha - 2

alpha^3 - 3*alpha^2 + 4*alpha - 1

-alpha^3 + 3*alpha^2 - 4*alpha + 2

-alpha^3 + 3*alpha^2 - 4*alpha + 1

-1

## Fixed field w.r.t. tau in K

In [5]:
K = D63.field
z = D63.root
tau = D63.tau

invs = [(1/z^i).polynomial() for i in range(4)]; print(invs)

var('c0, c1, c2, c3') 
cv = [c0, c1, c2, c3]

v_x = sum([cv[i]*x^i for i in range(4)])
v_inv = sum([cv[i]*invs[i] for i in range(4)]) #c0+c1*invs[1]+c2*invs[2]+c3*invs[3]
v_inv_collected = v_inv.collect(x)
display(v_inv, v_inv_collected)

equations = [cv[k] == v_inv_collected.coefficient(x^k) for k in range(1,4)]
print(equations)
solution = solve(equations, c1, c2, c3)
print(solution)

[1, -x^3 + 3*x^2 - 5*x + 3, -3*x^3 + 8*x^2 - 12*x + 4, -4*x^3 + 9*x^2 - 12*x]


-(x^3 - 3*x^2 + 5*x - 3)*c1 - (3*x^3 - 8*x^2 + 12*x - 4)*c2 - (4*x^3 - 9*x^2 + 12*x)*c3 + c0

-(c1 + 3*c2 + 4*c3)*x^3 + (3*c1 + 8*c2 + 9*c3)*x^2 - (5*c1 + 12*c2 + 12*c3)*x + c0 + 3*c1 + 4*c2

[c1 == -5*c1 - 12*c2 - 12*c3, c2 == 3*c1 + 8*c2 + 9*c3, c3 == -c1 - 3*c2 - 4*c3]
[
[c1 == 4*r1, c2 == -3*r1, c3 == r1]
]


In [6]:
w = v_x.subs(solution[0])
tau_w = sum(w.coefficients(x)[k][0]*invs[k] for k in range(4))

display(w, tau_w)
print(f"w == tau(w) ? --> {bool(w == tau_w)}")

r1*x^3 - 3*r1*x^2 + 4*r1*x + c0

-(4*x^3 - 9*x^2 + 12*x)*r1 + 3*(3*x^3 - 8*x^2 + 12*x - 4)*r1 - 4*(x^3 - 3*x^2 + 5*x - 3)*r1 + c0

w == tau(w) ? --> True


## C_K

In [7]:
p = (0*x^0)+x; display(p)
p.degree(x)

x

1

In [8]:
prod = v_x*v_inv
t = var('t'); f = D63.char_poly.subs({t:x}); display(f)

x^4 - 3*x^3 + 5*x^2 - 3*x + 1

In [9]:
mpoly = prod

while mpoly.degree(x) >=4:
    mpoly = mpoly.collect(x)
    max_deg = mpoly.degree(x); #display(max_deg)
    max_coeff = mpoly.coefficient(x^max_deg)
    #---
    mpoly -= max_coeff*x^max_deg
    mpoly += max_coeff*x^(max_deg-4)*(-1)*(f-x^4)
    #display(mpoly)

display(mpoly)

-(c0*c1 - 3*c1^2 + 3*c0*c2 - 8*c1*c2 - 3*c2^2 + 3*c0*c3 - 9*c1*c3 - 8*c2*c3 - 3*c3^2)*x^3 + (3*c0*c1 - 5*c1^2 + 9*c0*c2 - 12*c1*c2 - 5*c2^2 + 9*c0*c3 - 11*c1*c3 - 12*c2*c3 - 5*c3^2)*x^2 - (3*x^3 - 5*x^2 + 3*x - 1)*(c1^2 + 3*c1*c2 + c2^2 + 4*c1*c3 + 3*c2*c3 + c3^2) + c0^2 + 3*c0*c1 + 4*c0*c2 - (4*c0*c1 - 3*c1^2 + 12*c0*c2 - 5*c1*c2 - 3*c2^2 + 12*c0*c3 - 5*c2*c3 - 3*c3^2)*x

In [10]:
for c in mpoly.coefficients(x):
    display(c)

print(bool(mpoly.coefficient(x^3)*(-3) == mpoly.coefficient(x^2)), bool(mpoly.coefficient(x^3)*(4) == mpoly.coefficient(x^1)))

[c0^2 + 3*c0*c1 + c1^2 + 4*c0*c2 + 3*c1*c2 + c2^2 + 4*c1*c3 + 3*c2*c3 + c3^2,
 0]

[-4*c0*c1 - 12*c0*c2 - 4*c1*c2 - 12*c0*c3 - 12*c1*c3 - 4*c2*c3, 1]

[3*c0*c1 + 9*c0*c2 + 3*c1*c2 + 9*c0*c3 + 9*c1*c3 + 3*c2*c3, 2]

[-c0*c1 - 3*c0*c2 - c1*c2 - 3*c0*c3 - 3*c1*c3 - c2*c3, 3]

True True


In [11]:
cc0 = mpoly.coefficients(x)[0][0]
cc3 = mpoly.coefficients(x)[3][0]
display(cc0, cc3)

c0^2 + 3*c0*c1 + c1^2 + 4*c0*c2 + 3*c1*c2 + c2^2 + 4*c1*c3 + 3*c2*c3 + c3^2

-c0*c1 - 3*c0*c2 - c1*c2 - 3*c0*c3 - 3*c1*c3 - c2*c3

In [12]:
X, Y = var('X', 'Y')

cc0 = mpoly.coefficients(x)[0][0]
cc3 = mpoly.coefficients(x)[3][0]
solve([cc0==1, cc3==1], c0, c1, c2, c3)


KeyboardInterrupt



KeyboardInterrupt: 

In [ ]:
display(((c0+c1+c2+c3)^2-cc0).factor(), cc3)

In [ ]:
eq = (c0+c1+c2+c3)^2-(c0-c1)*(c2-c3); display(eq == eq.expand())
display((eq-(cc0+cc3)).factor())

In [ ]:
display(((c0^2+c1^2+c2^2+c3^2)-cc0-3*cc3).factor())

## Annulus twists

In [6]:
def inv(s):
    return s[::-1].swapcase()
def conj(x,y):
    return y + x + inv(y) if x else ''
#----------
def A(phi, n):
    if n >= 0:
        return 'C'*n + conj('a'*n, phi)
    else:
        return 'c'*abs(n) + inv(conj('a'*abs(n), phi))

In [7]:
phi = 'ecb' #'ECB' #
#----------
def A63(n:int):
    return A(phi, n) + m63        

In [14]:
md_data(A63(0)).show()


-- Symp. Matrix --


[ 0  1 -1 -1]
[-1  1  0  0]
[ 1 -1  1  0]
[ 0  0  1  1]


-- Number field --


Number Field in alpha with defining polynomial t^4 - 3*t^3 + 5*t^2 - 3*t + 1


-- basis of OK --


...

1


-- basis of U --


[alpha^3 - 3*alpha^2 + 4*alpha - 1, -alpha^3 + 2*alpha^2 - 3*alpha]


-- basis of C --


[-alpha^3 + 3*alpha^2 - 4*alpha + 2, alpha^3 - 3*alpha^2 + 4*alpha - 2]

In [8]:
R = 5
gens = []
for i, j in itertools.combinations(range((-1)*R, R+1), 2):
    at_pair = [A63(k) for k in [i,j]]
    g_pair = [md_data(m).generator for m in at_pair]
    val = g_pair[0]/g_pair[1]
    gens.append(((i,j), g_pair, (val, 1/val), (val in C63) or (1/val in C63)))

[display(g) for g in gens if g[0][0]+g[0][1]==-1];

((-5, 4),
 [1204*alpha^3 - 3612*alpha^2 + 4816*alpha + 3311,
  -2405*alpha^3 + 7215*alpha^2 - 9620*alpha + 13949],
 (51471/231361*alpha^3 - 154413/231361*alpha^2 + 205884/231361*alpha + 28294/231361,
  -82251/90601*alpha^3 + 246753/90601*alpha^2 - 329004/90601*alpha + 291967/90601),
 False)

((-4, 3),
 [291*alpha^3 - 873*alpha^2 + 1164*alpha + 485,
  -724*alpha^3 + 2172*alpha^2 - 2896*alpha + 3439],
 (7469/32761*alpha^3 - 22407/32761*alpha^2 + 29876/32761*alpha - 97/32761,
  -13937/9409*alpha^3 + 41811/9409*alpha^2 - 55748/9409*alpha + 41630/9409),
 False)

((-3, 2),
 [38*alpha^3 - 114*alpha^2 + 152*alpha + 19,
  -147*alpha^3 + 441*alpha^2 - 588*alpha + 539],
 (475/2401*alpha^3 - 1425/2401*alpha^2 + 1900/2401*alpha - 304/2401,
  -1225/361*alpha^3 + 3675/361*alpha^2 - 4900/361*alpha + 2891/361),
 False)

((-2, 1),
 [alpha^3 - 3*alpha^2 + 4*alpha - 1, -14*alpha^3 + 42*alpha^2 - 56*alpha + 35],
 (3/49*alpha^3 - 9/49*alpha^2 + 12/49*alpha - 5/49,
  -21*alpha^3 + 63*alpha^2 - 84*alpha + 28),
 False)

((-1, 0),
 [-1, -alpha^3 + 3*alpha^2 - 4*alpha + 1],
 (-alpha^3 + 3*alpha^2 - 4*alpha + 2, alpha^3 - 3*alpha^2 + 4*alpha - 1),
 True)

In [10]:
gens = []
i, j = (0,1) #(-2, 1)

at_pair = inv(A(phi, i) + m63), A(phi, j) + m63
print(at_pair)

g_pair = [md_data(m).generator for m in at_pair]
val = g_pair[0]/g_pair[1]

U = md_data(at_pair[0]).UnitGroup; print(U)
print(val in U)

gens.append(((i,j), g_pair, (val, 1/val), (val in C63) or (1/val in C63)))

[display(g) for g in gens]; # if g[0][0]+g[0][1]==-1];

('AcBd', 'CecbaBCEDbCa')
Unit group with structure C6 x Z of Number Field in alpha with defining polynomial t^4 - 3*t^3 + 5*t^2 - 3*t + 1
False


((0, 1),
 [alpha^3 - 3*alpha^2 + 4*alpha - 1, -14*alpha^3 + 42*alpha^2 - 56*alpha + 35],
 (3/49*alpha^3 - 9/49*alpha^2 + 12/49*alpha - 5/49,
  -21*alpha^3 + 63*alpha^2 - 84*alpha + 28),
 False)

In [17]:
mcs = []
evs = []
gens = []
for n in range(-1,5): #[-1,0]: #[1,-2]: #
    mc = A63(n)
    Dmc = md_data(mc)
    print(f"-- A^{n}(6_3) --")
    display(mc, Dmc.matrix, Dmc.eigen_vector, Dmc.generator) #, Dmc.char_poly)
    print("\n")
    #---
    if not Dmc.matrix*Dmc.eigen_vector == Dmc.root*Dmc.eigen_vector:
        print(f"What's happened?")
    
    #---
    mcs.append(mc)
    evs.append(Dmc.eigen_vector)
    gens.append(Dmc.generator)

val = gens[1]/gens[0]
print(f"-----\n{mcs}: conjugate?  -->  {conjugate_check(*mcs, rng=40)}")
display((val, 1/val))
print(f"-----\n{val} in setC?  -->  {val in C63}")
print(f"-----\n{1/val} in setC?  -->  {1/val in C63}")

-- A^-1(6_3) --


'cecbABCEDbCa'

[ 2  0 -1 -2]
[ 1  0  0 -1]
[ 1 -1  1  0]
[ 1  0  0  0]

(1, 2*alpha^3 - 5*alpha^2 + 7*alpha - 1, 2*alpha^3 - 6*alpha^2 + 9*alpha - 4, -alpha^3 + 3*alpha^2 - 5*alpha + 3)

-1

...

'CCCCecbaaaaBCEDbCa'

[-8  5 -1  3]
[-9  5  0  4]
[ 1 -1  1  0]
[-4  0  5  5]

(481, 77*alpha^3 - 230*alpha^2 + 407*alpha + 324, 97*alpha^3 - 271*alpha^2 + 444*alpha - 254, -96*alpha^3 + 293*alpha^2 - 370*alpha + 658)

-2405*alpha^3 + 7215*alpha^2 - 9620*alpha + 13949

TypeError: conjugate_check() got multiple values for argument 'rng'

# Scratch

In [18]:
basis =  vector([1, z, z^2, z^3])

def num_to_vec(num):
    return vector([num.polynomial().coefficient(i) for i in range(4)])
    
def vec_to_mat(v):
    return matrix([num_to_vec(e) for e in v]).transpose()
    
def mat_to_vec(mat):
    return basis*(mat.transpose())

    
vec = evs[0]
mmtx = vec_to_mat(vec)
mvec = mat_to_vec(mmtx)
display(vec, mmtx, mvec == vec)

(1, 2*alpha^3 - 5*alpha^2 + 7*alpha - 1, 2*alpha^3 - 6*alpha^2 + 9*alpha - 4, -alpha^3 + 3*alpha^2 - 5*alpha + 3)

[ 1 -1 -4  3]
[ 0  7  9 -5]
[ 0 -5 -6  3]
[ 0  2  2 -1]

False

In [ ]:
A_alpha = vec_to_mat(z*basis).transpose(); display(z*basis, A_alpha)

num = vec[1]

display((num, z*num))
display((num_to_vec(num), num_to_vec(z*num), A_alpha*num_to_vec(num)))

M = A_alpha*mmtx; display(M)
#display(mat_to_vec(M), z*vec)